<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/err/cross_section_momentum/append_to_Duckdb_dbase%20from_%20pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import os
import zipfile
import shutil
import pandas as pd
import numpy as np

source_folder = "/content"           # Folder containing ZIP files
levelone_folder = "/content/levelone"

os.makedirs(levelone_folder, exist_ok=True)

def extract_all_zips(source, destination):

    # First copy all zip files to levelone
    for file in os.listdir(source):
        if file.endswith(".zip"):
            shutil.copy(os.path.join(source, file), destination)

    # Now recursively extract inside levelone
    while True:
        zip_found = False

        for root, dirs, files in os.walk(destination):
            for file in files:
                if file.endswith(".zip"):
                    zip_found = True
                    zip_path = os.path.join(root, file)

                    with zipfile.ZipFile(zip_path, 'r') as z:
                        z.extractall(root)

                    os.remove(zip_path)
                    print(f"Extracted and removed: {file}")

        if not zip_found:
            break

    print("All ZIP files fully extracted into levelone.")


extract_all_zips(source_folder, levelone_folder)

Extracted and removed: Reports-Archives-Multiple-23042025.zip
Extracted and removed: Reports-Archives-Multiple-13032025.zip
Extracted and removed: Reports-Archives-Multiple-19082025.zip
Extracted and removed: Reports-Archives-Multiple-07082024.zip
Extracted and removed: Reports-Archives-Multiple-22082025.zip
Extracted and removed: Reports-Archives-Multiple-03122024.zip
Extracted and removed: Reports-Archives-Multiple-13112024.zip
Extracted and removed: Reports-Archives-Multiple-12022025.zip
Extracted and removed: Reports-Archives-Multiple-15092025.zip
Extracted and removed: Reports-Archives-Multiple-13092024.zip
Extracted and removed: Reports-Archives-Multiple-18082025.zip
Extracted and removed: Reports-Archives-Multiple-21072025.zip
Extracted and removed: Reports-Archives-Multiple-23102024.zip
Extracted and removed: Reports-Archives-Multiple-12092024.zip
Extracted and removed: Reports-Archives-Multiple-23092025.zip
Extracted and removed: Reports-Archives-Multiple-19072024.zip
Extracte

In [20]:
full_data.columns

Index(['TradDt', 'BizDt', 'Sgmt', 'Src', 'FinInstrmTp', 'FinInstrmId', 'ISIN',
       'TckrSymb', 'SctySrs', 'XpryDt', 'FininstrmActlXpryDt', 'StrkPric',
       'OptnTp', 'FinInstrmNm', 'OpnPric', 'HghPric', 'LwPric', 'ClsPric',
       'LastPric', 'PrvsClsgPric', 'UndrlygPric', 'SttlmPric', 'OpnIntrst',
       'ChngInOpnIntrst', 'TtlTradgVol', 'TtlTrfVal', 'TtlNbOfTxsExctd',
       'SsnId', 'NewBrdLotQty', 'Rmks', 'Rsvd1', 'Rsvd2', 'Rsvd3', 'Rsvd4'],
      dtype='object')

In [21]:
# Keep only Cash Market segment
full_data = full_data[full_data["Sgmt"] == "CM"]

# Keep only Equity series
full_data = full_data[full_data["SctySrs"] == "EQ"]
'''
# Select required columns
full_data = full_data[[
    "TckrSymb",
    "TradDt",
    "OpnPric",
    "HghPric",
    "LwPric",
    "ClsPric",
    "TtlTradgVol"
]]

# Rename to clean structure
full_data.columns = [
    "symbol",
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume"
]
# Fix date

full_data["date"] = pd.to_datetime(full_data["date"])

print("Equity data cleaned.")
print(full_data.head())
'''

'\n# Select required columns\nfull_data = full_data[[\n    "TckrSymb",\n    "TradDt",\n    "OpnPric",\n    "HghPric",\n    "LwPric",\n    "ClsPric",\n    "TtlTradgVol"\n]]\n\n# Rename to clean structure\nfull_data.columns = [\n    "symbol",\n    "date",\n    "open",\n    "high",\n    "low",\n    "close",\n    "volume"\n]\n# Fix date\n\nfull_data["date"] = pd.to_datetime(full_data["date"])\n\nprint("Equity data cleaned.")\nprint(full_data.head())\n'

In [22]:
import duckdb

# -----------------------------
# CONFIG
# -----------------------------
db_path = "/content/nifty_200_data.duckdb"
table_name = "equities"   # Change if your table name is different

# -----------------------------
# 1. Connect to DuckDB
# -----------------------------
con = duckdb.connect(db_path)

# -----------------------------
# 2. Create Table If Not Exists
# -----------------------------
con.execute(f"""
CREATE TABLE IF NOT EXISTS {table_name} AS
SELECT * FROM full_data WHERE 1=0
""")

# -----------------------------
# 3. Register pandas dataframe
# -----------------------------
con.register("temp_df", full_data)

# -----------------------------
# 4. Append Only New Rows
# -----------------------------
con.execute(f"""
INSERT INTO {table_name}
SELECT * FROM temp_df
WHERE (TckrSymb, TradDt) NOT IN (
    SELECT TckrSymb, TradDt FROM {table_name}
)
""")

print("Append complete.")

Append complete.
